# TL_Base_Model - 2023


# Importing Libraries


In [1]:
# import logging

# logging.disable(logging.WARNING)
from TL_Packages import *
from datetime import datetime
from sklearn.model_selection import KFold, train_test_split
import tensorflow as tf

import keras
from keras.models import load_model

import numpy as np
import os
import pandas as pd

pd.set_option("display.max_columns", None)

# Point of Note - TensorFlow namespace is used for data modelling. However, the models are being saved using the Keras namespace. Due to which loading should be done using the Keras namespace only
# In Future codes, please use the TensorFlow namespace for both, but that will require some changes in the code or models can loaded and resaved in TensorFlow namespace as well.

In [ ]:
# Tensorflow Configuration
import multiprocessing

# Get the total number of CPU cores available
total_cores = multiprocessing.cpu_count()

# Calculate 75% of available cores (rounded down)
cores_to_use = int(total_cores * 0.90)

# Ensure at least 1 core is used
cores_to_use = max(1, cores_to_use)

# Set the environment variable
os.environ["LOKY_MAX_CPU_COUNT"] = str(cores_to_use)

print(f"Using {cores_to_use} out of {total_cores} available CPU cores (90%)")

# Add after TensorFlow import
tf.config.threading.set_inter_op_parallelism_threads(cores_to_use // 2)
tf.config.threading.set_intra_op_parallelism_threads(cores_to_use // 2)

# Version Checks


In [ ]:
# SET DIRECTORY
execution_path = "../../../"

# CHECK FOR CORRECT KERAS AND PANDAS VERSIONS
print("keras.__version__ = ", keras.__version__)  # 2.12.0

# Error will occur if pandas greater than specified due to loss of backward compatibility
# https://stackoverflow.com/questions/75953279/modulenotfounderror-no-module-named-pandas-core-indexes-numeric-using-metaflo
# pip install "pandas<2.0.0"
print("pd.__version__ = ", pd.__version__)  # 1.4.0 should be <2
print("np.__version__ = ", np.__version__)
print("tf.__version__ = ", tf.__version__)  # 2.10.0
# [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
print("tf.config.list_physical_devices('GPU') = ", tf.config.list_physical_devices())
print("tf.test.is_built_with_cuda() = ", tf.test.is_built_with_cuda())  # True
device = "cuda" if tf.test.is_built_with_cuda() else "cpu"
device = "cpu"  # running only on cpu
print("device = ", device)

# Settings - To Control Execution


In [2]:
# SETTING GLOBALS AND FILEPATHS

# Data to use:
source_quantile = "0.2"
source_file_path = Path(
    f"{execution_path}Input_Data/Og_Data/labeled_normal_data/2all_pids_q={source_quantile}.npy"
)
print("source_file_path", source_file_path)
# Possible Values: "TL_1_Frozen", "TL_2_Frozen", "TL_3_Frozen", "TL_All_Frozen", "TL_All_Unfrozen"
Architecture_GROUP = "TL_All_Unfrozen"

# CHANGE ACCORDING TO YOUR ALGO
# Possible Values: ADKNN , GammaSmoteBoost,  <<-- Other Teams append their Algorithms here -->>
augmentation_algo = "GammaSmoteBoost"
augmentation_algo = "ADKNN"
augmentation_algo = "SMOTEBoostCC"
augmentation_algo = "SDDSMOTEBoostCC"

# When True only small sample of patients will run for testing purpose else All Patients as per parameters
bool_testing = False

# WHEN FALSE, TARGET DATA IS NOT REMOVED FROM ALL PATIENTS DATA
# False for Generalized model, True for Patient-wise Pop Models
bool_remove_target = False

# When True will drop Age, BMI and Sex columns too else will keep them too
bool_drop_ABS = True

# When True will execute population_model else do not execute
# Note: population_model is generally executed only once and its output is reused multiple times
bool_execute_pop_model = False

# Number of Folds for running Pop Model
# Possible values : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
num_pop_folds = 5

# When True will execute individual_model else do not execute
bool_execute_ind_model = True

# Number of Folds for running Ind Model
# Possible values : [3, 4, 5]
num_ind_folds = 3

## Use True if you choose to not have cv for your individual models
bool_remove_ind_cv = False

# Change output folder during each run to not overwrite previous outputs\
# Should not include start or end '/'
output_folder = "2025_testing"

if bool_drop_ABS == False:
    output_folder += "_ABS"

RESULT_DIR = Path(f"{execution_path}/Output_Results/{output_folder}/")
print("RESULT_DIR", RESULT_DIR)

minority_variable = 0
CLASS_VAR = "class"


POPULATION_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]
PATIENT_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]

""" Removing SB-081 due to less data """
""" Removing SB-033 due to less data """
""" Removing SB-089 due to less data """

if bool_remove_target == True:
    pop_target_grp = [
        "SB-001",
        "SB-003",
        "SB-008",
        "SB-011",
        "SB-012",
        "SB-022",
        "SB-023",
        "SB-025",
        "SB-028",
        "SB-037",
        "SB-043",
        "SB-056",
        "SB-059",
        "SB-060",
        "SB-070",
        "SB-071",
        "SB-073",
        "SB-078",
        "SB-079",
        "SB-080",
        "SB-082",
        "SB-083",
        "SB-112",
    ]
else:
    pop_target_grp = ["source"]

# Used for individual model
ind_target_grp = [
    "SB-001",
    "SB-003",
    "SB-008",
    "SB-011",
    "SB-012",
    "SB-022",
    "SB-023",
    "SB-025",
    "SB-028",
    "SB-037",
    "SB-043",
    "SB-056",
    "SB-059",
    "SB-060",
    "SB-070",
    "SB-071",
    "SB-073",
    "SB-078",
    "SB-079",
    "SB-080",
    "SB-082",
    "SB-083",
    "SB-112",
]

EVALUATION_METRICS = [
    "Weighted Accuracy",
    "Sensitivity/Recall",
    "Specificity",
    "Precision_class0",
    "Precision_class1",
    "Precision_avg",
    "F1_class0",
    "F1_class1",
    "F1_avg",
    "auc_roc_score",
    "False_Discovery_Rate",
    "False_Negative_Rate",
    "False_Omission_Rate",
    "False_Positive_Rate",
    "Jaccard",
]

source_file_path ..\..\..\Input_Data\Og_Data\labeled_normal_data\2all_pids_q=0.2.npy
RESULT_DIR ..\..\..\Output_Results\2025_testing


In [ ]:
def runtime():
    """
    Print current time: "Run-time: <hour> o'clock and <minute> minutes"
    """
    print(
        "Runtime:",
        datetime.now().hour,
        "o' clock and",
        datetime.now().minute,
        "minutes",
    )

## Recreating patient dataset numpy file using latest numpy

In [3]:
import os
import pandas as pd
import numpy as np
from glob import glob

# Define paths
base_path = "../../../Input_Data/Og_data/labeled_normal_data/q=0.2/"
output_file = "../../../Input_Data/Og_Data/labeled_normal_data/2all_pids_q=0.2.npy"

# Create a dictionary to store all patient data
all_patient_data = {}

# Find all patient folders
patient_folders = glob(os.path.join(base_path, "SB-*"))

# Loop through each patient folder
for patient_folder in patient_folders:
    # Extract patient ID from folder name
    patient_id = os.path.basename(patient_folder)

    # Construct path to the normalized CSV file
    csv_file = os.path.join(patient_folder, f"{patient_id}_normalized_AM.csv")

    # Check if file exists
    if os.path.exists(csv_file):
        # Read the CSV file into a DataFrame
        df = pd.read_csv(csv_file)

        # Store the DataFrame in the dictionary with patient ID as key
        all_patient_data[patient_id] = df
        print(f"Processed {patient_id}: {df.shape[0]} rows")
    else:
        print(f"Warning: CSV file not found for {patient_id}")

# Save the dictionary as a numpy file
np.save(output_file, all_patient_data)

print(f"\nSaved data for {len(all_patient_data)} patients to {output_file}")

# Verify the saved data
loaded_data = np.load(output_file, allow_pickle=True).item()
print(f"Verification: Loaded {len(loaded_data)} patients from {output_file}")

# Print sample from first patient
sample_patient = next(iter(loaded_data))
print(f"\nSample data from {sample_patient}:")
print(loaded_data[sample_patient].head(3))

Processed SB-001: 188 rows
Processed SB-003: 180 rows
Processed SB-008: 178 rows
Processed SB-011: 210 rows
Processed SB-012: 194 rows
Processed SB-022: 204 rows
Processed SB-023: 194 rows
Processed SB-025: 117 rows
Processed SB-028: 92 rows
Processed SB-033: 53 rows
Processed SB-037: 210 rows
Processed SB-043: 191 rows
Processed SB-056: 191 rows
Processed SB-059: 203 rows
Processed SB-060: 88 rows
Processed SB-070: 202 rows
Processed SB-071: 124 rows
Processed SB-073: 161 rows
Processed SB-078: 153 rows
Processed SB-079: 142 rows
Processed SB-080: 181 rows
Processed SB-081: 34 rows
Processed SB-082: 137 rows
Processed SB-083: 206 rows
Processed SB-089: 49 rows
Processed SB-112: 180 rows

Saved data for 26 patients to ../../../Input_Data/Og_Data/labeled_normal_data/2all_pids_q=0.2.npy
Verification: Loaded 26 patients from ../../../Input_Data/Og_Data/labeled_normal_data/2all_pids_q=0.2.npy

Sample data from SB-001:
       id        date  age  sex     BMI  income  road_dist  cooking  \
0

# Importing Data


In [4]:
# CONVERT Data TO NUMPY ARRAYS
data_ndarr = np.load(source_file_path, allow_pickle=True)
print("data_ndarr type :", type(data_ndarr))

datadict = data_ndarr.item()
print("datadict type :", type(datadict))

data_ndarr type : <class 'numpy.ndarray'>
datadict type : <class 'dict'>


# Processing data for Usage


In [5]:
# DROPPING UN-USED COLUMNS

p_ids = datadict.keys()
sample_size_dict = {}
print("patients: n =", len(p_ids), end="\n\n")

for p_id in p_ids:
    df = datadict[p_id]

    if bool_drop_ABS:
        df.drop(["id", "date", "am_pef_org", "BMI", "sex", "age"], axis=1, inplace=True)
    else:
        df.drop(["id", "date", "am_pef_org"], axis=1, inplace=True)

    df = np.asarray(df).astype(np.float32)
    print(p_id, "shape:", df.shape)
    sample_size_dict[p_id] = df.shape[0]


display(datadict[p_id].head(5))

"""
Dropped columns above
"""
"""
`date` was dropped above, because it would throw `ValueError: could not convert string to float: '2017-11-01'`
"""
"""
`id` and `am_pef_org` were dropped because neither are explanatory variables.
"""
"""
(Note: `am_pef_org` is the UNSHIFTED, NON-NORMALIZED value for a person's morning exposure. It is relatively raw, and should almost exclusively be used for classification purposes)
"""

patients: n = 26

SB-001 shape: (188, 28)
SB-003 shape: (180, 28)
SB-008 shape: (178, 28)
SB-011 shape: (210, 28)
SB-012 shape: (194, 28)
SB-022 shape: (204, 28)
SB-023 shape: (194, 28)
SB-025 shape: (117, 28)
SB-028 shape: (92, 28)
SB-033 shape: (53, 28)
SB-037 shape: (210, 28)
SB-043 shape: (191, 28)
SB-056 shape: (191, 28)
SB-059 shape: (203, 28)
SB-060 shape: (88, 28)
SB-070 shape: (202, 28)
SB-071 shape: (124, 28)
SB-073 shape: (161, 28)
SB-078 shape: (153, 28)
SB-079 shape: (142, 28)
SB-080 shape: (181, 28)
SB-081 shape: (34, 28)
SB-082 shape: (137, 28)
SB-083 shape: (206, 28)
SB-089 shape: (49, 28)
SB-112 shape: (180, 28)


,income,road_dist,cooking,y_am_pef,tempin,humidin,pm25in,co2in,tempdiffin,humidiffin,pm25diffin,pm10,pm25,o3,no2,co,so2,temp,windsd,humid,varp,dewpt,airp,seap,solrhr,solramnt,grdt,class
0,0.857,0.0,0.167,0.558,0.216,0.513,0.358,0.678,0.562,0.364,0.619,0.390,0.250,0.024,0.587,0.536,0.222,0.349,0.107,0.668,0.174,0.471,0.822,0.819,0.165,0.003,0.206,0.0
1,0.857,0.0,0.167,0.006,0.299,0.521,0.411,0.700,0.436,0.672,0.616,0.446,0.285,0.000,0.783,0.732,0.222,0.473,0.171,0.388,0.190,0.491,0.676,0.670,0.603,0.165,0.383,0.0
2,0.857,0.0,0.167,0.078,0.420,0.713,0.372,0.918,0.469,0.821,0.734,0.466,0.155,0.190,0.239,0.262,0.111,0.474,0.515,0.359,0.197,0.481,0.519,0.518,0.485,0.104,0.346,1.0
3,0.857,0.0,0.167,0.428,0.438,0.468,0.224,0.762,0.560,0.490,0.794,0.211,0.011,0.143,0.196,0.110,0.111,0.404,0.338,0.527,0.175,0.467,0.596,0.596,0.491,0.141,0.339,0.0
4,0.857,0.0,0.167,0.209,0.462,0.385,0.199,0.502,0.612,0.529,0.725,0.250,0.057,0.095,0.348,0.245,0.111,0.366,0.316,0.444,0.141,0.419,0.765,0.762,0.433,0.135,0.281,1.0


"\n(Note: `am_pef_org` is the UNSHIFTED, NON-NORMALIZED value for a person's morning exposure. It is relatively raw, and should almost exclusively be used for classification purposes)\n"

# Function Definition - split_data (used by both - population_model and individual_model)


In [ ]:
def split_data(
    target, data, bool_remove_target, transfer=False
) -> (pd.DataFrame, pd.DataFrame):
    """
    Creates a combined data set from all the ids in the given dictionary,
    If transfer==false, returns just target df
    If transfer==True and bool_remove_target==True, returns all but the target df.
    If transfer==True and bool_remove_target==False, return all patients’ data

    :param target: Who the target is for prediction.
    :param data: Dict of df's for prediction
    :return X: Predictive labels
    :return Y: Dependent label
    """
    dataset = pd.DataFrame()
    if transfer == True:
        if bool_remove_target == True:
            dataset = pd.concat(
                [data[key].copy() for key in data.keys() if key != target]
            )
        else:
            dataset = pd.concat([data[key].copy() for key in data.keys()])

    else:
        dataset = data[target].copy()
    X = dataset.drop([CLASS_VAR], axis=1)
    Y = dataset.loc[:, CLASS_VAR:CLASS_VAR]
    return X, Y

In [ ]:
import gc
import psutil
import os


# Memory monitoring function
def check_memory_usage(threshold_percent=80):
    """Monitor memory usage and return True if above threshold"""
    memory = psutil.virtual_memory()
    memory_percent = memory.percent
    print(f"Current memory usage: {memory_percent:.2f}%")
    return memory_percent > threshold_percent

# Function Definition - data_augmentation (used by both - population_model and individual_model)


In [ ]:
# CALLS THE AUGMENTATION FUNCTION TO BALANCE DATASET
def data_augmentation(
    augmentation_algo,
    train_dataset,
    classIndex,
    minorityLabel,
    printDebug=False,
):
    """
    Augments the given df via the given technique

    If we want other to be able to plug in their methods here, they need to be able to understand what
    is fed into their function call and what is returned.

    :param augmentation_algo: String name of technique to use
    :param train_dataset: df to be augmented. Has both the X and Y labels.
                        Complete dataset except only train split. Already dropped:
                        ["id", "date", "am_pef_org", "BMI", "sex", "age"]
    :param classIndex: Label of class with minority/majority split
    :param minorityLabel: Which y value of classIndex is the minority class
    :param printDebug: Passed to technique for debug printing
    :param upSampling: unused yet... further implementation TBD
    :return: Augmented data set. Same as train_dataset except has synthetic rows included. Should be balanced.
    """
    if augmentation_algo == "ADKNN":
        train_dataset = balance_method.ADKNN(
            train_dataset,
            classIndex=classIndex,
            minorityLabel=minorityLabel,
            printDebug=printDebug,
        )
    if augmentation_algo == "AutoEncoder_1":
        train_dataset = balance_method.generate_autoencoder(
            train_dataset,
            class_var=classIndex,
            minority_var=minorityLabel,
            printDebug=printDebug,
        )
    if augmentation_algo == "GammaSmoteBoost":
        train_dataset = balance_method.GammaSMOTEBoostCC(
            train_dataset,
            numIterations=5,
            printDebug=printDebug,
        )
    if augmentation_algo == "SDDSMOTEBoostCC":
        train_dataset = balance_method.SDDSMOTEBoostCC(
            train_dataset,
            printDebug=printDebug,
        )
    if augmentation_algo == "SMOTEBoostCC":
        train_dataset = balance_method.SMOTEBoostCC(
            train_dataset,
            numIterations=5,
            printDebug=printDebug,
        )
        print(f"augmented data size: {train_dataset.shape}")
    # <<-- PASTE OTHER ALGO CALLS BELOW CONDITIONALLY -->>

    print("~~~~~~~ value_counts: ~~~~~~~")
    print("SHOULD BE BALANCED NOW:\n", train_dataset[classIndex].value_counts())
    return train_dataset

# Checking Variable values before execution


In [ ]:
print(
    "Len:",
    len(POPULATION_EPOCH_SIZE_LIST),
    "\n",
    "POPULATION_EPOCH_SIZE_LIST:",
    POPULATION_EPOCH_SIZE_LIST,
)

print(
    "Len:",
    len(PATIENT_EPOCH_SIZE_LIST),
    "\n",
    "PATIENT_EPOCH_SIZE_LIST:",
    PATIENT_EPOCH_SIZE_LIST,
)

print(
    "Len:",
    len(pop_target_grp),
    "\n",
    "pop_target_grp:",
    pop_target_grp,
)

print(
    "Len:",
    len(ind_target_grp),
    "\n",
    "ind_target_grp:",
    ind_target_grp,
)

# Function Definition - find_pop_model (used by individual_model)


In [ ]:
def find_pop_model(
    target_id, RESULT_DIR, Architecture_GROUP, POPULATION_EPOCH_SIZE_LIST
):
    """
    Returns all population model names found in target directory
    :param target_id: Participant ID of models to search for
    :param RESULT_DIR: Results directory
    :param Architecture_GROUP: Sub directory of results to search in.
    """
    pop_model_list = []
    pop_model_path = f"{str(RESULT_DIR)}/{Architecture_GROUP}/diagrams_n_population_models/{target_id}"
    # This finds a list of pop_model based on parameters provided
    for POPULATION_EPOCH_SIZE in POPULATION_EPOCH_SIZE_LIST:
        pop_model_list.extend(
            file_name
            for file_name in os.listdir(pop_model_path)
            if (
                file_name[-6:] == ".keras"
                and "_" + str(POPULATION_EPOCH_SIZE) + "_" in file_name
                and target_id in file_name
            )
        )
    return pop_model_list


if bool_testing:
    target_id = pop_target_grp[0]
    RESULT_DIR = Path(f"{execution_path}/Output_Results/{output_folder}/")
    pop_model_file_list = find_pop_model(
        target_id, RESULT_DIR, Architecture_GROUP, POPULATION_EPOCH_SIZE_LIST
    )
    display(pop_model_file_list)

# Function Definition - individual_model


In [ ]:
def individual_model(
    target_id,
    RESULT_DIR,
    Architecture_GROUP,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    datadict,
    bool_remove_target,
    EVALUATION_METRICS,
    bool_remove_ind_cv,
    num_ind_folds=3,
):
    """
    Transfer learning for individual target. Takes a previously trained source model and refines it for target_id.

    :param target_id: Participant ID to refine model on
    :param RESULT_DIR: Where to save results
    :param Architecture_GROUP: What type of architecture to use (string of how many layers are unfrozen)
    :param POPULATION_EPOCH_SIZE_LIST: Number of epochs of base model
    :param PATIENT_EPOCH_SIZE_LIST: Number of epochs to refine model with
    :param datadict: Dictionary of data to use
    :param bool_remove_target: Some confusion here, see targetSet comment below
    :param EVALUATION_METRICS: List of evaluation metrics to use
    :param bool_remove_ind_cv: Whether to remove individual cross-validation
    :param num_ind_folds: Number of k-folds for refinement
    :returns: Saves model to /diagrams_n_individual_models/{target_id}/
    """

    if bool_remove_ind_cv:
        num_ind_folds = 1
    else:
        if num_ind_folds < 3 or num_ind_folds > 5:
            print(
                f"num_ind_folds can only be set between 3 and 5, but current value is {num_ind_folds}"
            )
            num_ind_folds = 3 if num_ind_folds < 3 else 5
            print(f"num_ind_folds has been updated to {num_ind_folds}")

    runtime()

    patient_dir = Path(
        f"{RESULT_DIR}/{Architecture_GROUP}/diagrams_n_individual_models/{target_id}"
    )
    # Create directory for this patient
    if not (patient_dir.exists() and patient_dir.is_dir()):
        os.makedirs(patient_dir)

    X, Y = split_data(
        target=target_id,
        data=datadict,
        bool_remove_target=bool_remove_target,
        transfer=False,
    )

    # KFold requires n > 1, need special case for no k-fold
    if num_ind_folds > 1:
        kf = KFold(n_splits=num_ind_folds, shuffle=True).split(X.to_numpy())

    for fold in range(1, num_ind_folds + 1):
        print(f"Starting fold {fold} for patient {target_id}")

        gc.collect()
        # Clear TensorFlow session again
        tf.keras.backend.clear_session()

        # Make train/test sets
        if num_ind_folds == 1:
            X_train, X_test, Y_train, Y_test = train_test_split(
                X.to_numpy(), Y.to_numpy(), test_size=0.2, shuffle=True, random_state=42
            )
        else:
            train_index, test_index = next(kf)
            X_train, X_test = X.to_numpy()[train_index], X.to_numpy()[test_index]
            Y_train, Y_test = Y.to_numpy()[train_index], Y.to_numpy()[test_index]

        # Create train dataset as DataFrame
        df_X_train = pd.DataFrame(X_train, columns=X.columns.values)
        df_Y_train = pd.DataFrame(Y_train, columns=Y.columns.values)
        train_dataset = df_X_train.join(df_Y_train)

        print(f"Data augmentation starting for {target_id}, fold {fold}")
        train_dataset = data_augmentation(
            augmentation_algo,
            train_dataset,
            classIndex=CLASS_VAR,
            minorityLabel=minority_variable,
        )
        print(f"Data augmentation completed for {target_id}, fold {fold}")

        # Create DataFrame for this fold's results only
        fold_evaluation_results = pd.DataFrame(columns=EVALUATION_METRICS)

        X_train = train_dataset.drop(CLASS_VAR, axis=1).to_numpy()
        Y_train = train_dataset.loc[:, CLASS_VAR:CLASS_VAR].to_numpy()
        print(f"Fold {fold}: Training data size = {len(X_train)}")

        if bool_remove_target:
            pop_model_file_list = find_pop_model(
                target_id,
                RESULT_DIR,
                Architecture_GROUP,
                POPULATION_EPOCH_SIZE_LIST,
            )
        else:
            pop_model_file_list = find_pop_model(
                "source",
                RESULT_DIR,
                Architecture_GROUP,
                POPULATION_EPOCH_SIZE_LIST,
            )

        print(f"Found {len(pop_model_file_list)} population models for {target_id}")

        # Process each population model and patient epoch combination
        for pop_model_file in pop_model_file_list:
            for PATIENT_EPOCH_SIZE in PATIENT_EPOCH_SIZE_LIST:
                print(
                    f" ~~~~~ Patient: {target_id} ~~~~~ Fold: {fold} ~~~~~ Pop_Model_File: {pop_model_file} ~~~~~ PATIENT EPOCH SIZE: {PATIENT_EPOCH_SIZE} ~~~~~"
                )

                # Setup paths and tags
                if bool_remove_target:
                    pop_model_file_path = f"{str(RESULT_DIR)}/{Architecture_GROUP}/diagrams_n_population_models/{target_id}/{pop_model_file}"
                    VERSION_TAG = (
                        f"{pop_model_file[:-6]}_{PATIENT_EPOCH_SIZE}_IND_FOLD_{fold}"
                    )
                else:
                    pop_model_file_path = f"{str(RESULT_DIR)}/{Architecture_GROUP}/diagrams_n_population_models/source/{pop_model_file}"
                    VERSION_TAG = f"{pop_model_file[:-6]}_{target_id}-{PATIENT_EPOCH_SIZE}_IND_FOLD_{fold}"

                try:
                    # Create a new network for each model
                    net = network_tl(
                        X_train,
                        Y_train,
                        n_hidden=32,
                        learning_rate=0.001,
                        device=device,
                    )
                    net.set_Y_tind(Y_train)

                    print(f"Loading pretrained model from {pop_model_file_path}")
                    pretrained_model = load_model(pop_model_file_path, compile=False)

                    print(f"Creating individual model architecture")
                    net.make_ind(Architecture_GROUP, pretrained_model)

                    print(f"Retraining model with {PATIENT_EPOCH_SIZE} epochs")
                    net.retrain(X_train, Y_train, n_iters=PATIENT_EPOCH_SIZE)

                    # Save model immediately after training
                    ind_model_file_path = f"{str(RESULT_DIR)}/{Architecture_GROUP}/diagrams_n_individual_models/{target_id}/{VERSION_TAG}.keras"
                    print(f"Saving model to {ind_model_file_path}")
                    net.nn_ind.save(ind_model_file_path)

                    # Save training curve immediately
                    training_curve_file = f"{str(RESULT_DIR)}/{Architecture_GROUP}/diagrams_n_individual_models/{target_id}/{VERSION_TAG}_retraining_curve"
                    print(f"Saving training curve to {training_curve_file}")
                    net.save_training_curve(training_curve_file)

                    # Get test results
                    print(f"Testing model performance")
                    target_results, target_c_matrix = net.test(X_test, Y_test)

                    # Add this specific model's results to the fold results DataFrame
                    fold_evaluation_results.loc[VERSION_TAG] = list(target_results)

                    # Save the individual result immediately
                    if bool_remove_target:
                        output_path = os.path.join(
                            f"{RESULT_DIR}/{Architecture_GROUP}/diagrams_n_individual_models",
                            f"Ind_{Architecture_GROUP}_evaluation_metrics_TargetRemoved.csv",
                        )
                    else:
                        output_path = os.path.join(
                            f"{RESULT_DIR}/{Architecture_GROUP}/diagrams_n_individual_models",
                            f"Ind_{Architecture_GROUP}_evaluation_metrics.csv",
                        )

                    # Append this single result to the CSV
                    print(f"Saving evaluation metrics to {output_path}")
                    single_result_df = pd.DataFrame(
                        fold_evaluation_results.loc[[VERSION_TAG]]
                    )

                    if os.path.isfile(output_path):
                        single_result_df.to_csv(output_path, mode="a", header=False)
                    else:
                        single_result_df.to_csv(output_path, mode="a")

                    print(f"Completed model {VERSION_TAG}")

                    # Clean up resources
                    del pretrained_model
                    del net
                    gc.collect()
                    tf.keras.backend.clear_session()

                except Exception as e:
                    print(f"Error processing model {VERSION_TAG}: {str(e)}")
                    # Continue with next model even if this one fails
                    gc.collect()
                    tf.keras.backend.clear_session()
                    continue

        # Fold is completed
        print(f"Fold {fold} completed for patient {target_id}")

    print(f"All evaluation results saved for patient {target_id}")
    tf.keras.backend.clear_session()

# Executing - Individual Model


In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import psutil
import time


def check_memory_usage(threshold_percent=85):
    memory = psutil.virtual_memory()
    memory_percent = memory.percent
    print(f"Current memory usage: {memory_percent:.2f}%")
    return memory_percent > threshold_percent


# Create a wrapper function that checks memory before processing
def process_target_with_memory_check(target_id):
    try:
        # Wait until memory usage is below threshold
        while check_memory_usage(threshold_percent=85):
            print(
                f"Memory usage too high, waiting 5 minutes before processing {target_id}..."
            )
            time.sleep(300)  # Sleep for 5 minutes

        # Clear any existing TensorFlow state
        print(f"Starting processing for {target_id}")
        tf.keras.backend.clear_session()

        # Call individual model
        individual_model(
            target_id=target_id,
            RESULT_DIR=RESULT_DIR,
            Architecture_GROUP=Architecture_GROUP,
            POPULATION_EPOCH_SIZE_LIST=POPULATION_EPOCH_SIZE_LIST,
            PATIENT_EPOCH_SIZE_LIST=PATIENT_EPOCH_SIZE_LIST,
            datadict=datadict,
            bool_remove_target=bool_remove_target,
            EVALUATION_METRICS=EVALUATION_METRICS,
            bool_remove_ind_cv=bool_remove_ind_cv,
            num_ind_folds=num_ind_folds,
        )

        # Force garbage collection
        gc.collect()

        # Clear TensorFlow session again
        tf.keras.backend.clear_session()
        return f"Completed {target_id}"
    except Exception as e:
        return f"Error processing {target_id}: {str(e)}"


# Skip already processed patients
processed_ids = []  # Update this with any already processed IDs

if bool_testing is False:
    if bool_execute_ind_model:
        # Filter out already processed target IDs
        remaining_targets = [tid for tid in ind_target_grp if tid not in processed_ids]
        print(f"Processing {len(remaining_targets)} remaining targets")

        # Use fewer workers for TensorFlow tasks to avoid memory issues
        tf_workers = max(12, cores_to_use // 4)
        print(f"Using {tf_workers} worker processes")

        # Process targets sequentially with memory check
        if tf_workers == 1:
            for target_id in remaining_targets:
                result = process_target_with_memory_check(target_id)
                print(result)
        else:
            # Process targets in parallel but with controlled submission
            with ProcessPoolExecutor(max_workers=tf_workers) as executor:
                # Start with a limited batch of tasks
                futures = {}
                batch_size = min(tf_workers * 2, len(remaining_targets))

                # Submit initial batch
                for i in range(batch_size):
                    # Check memory before submitting
                    while check_memory_usage(threshold_percent=85):
                        print(
                            "Memory usage too high, waiting 5 minutes before submitting more tasks..."
                        )
                        time.sleep(300)  # Sleep for 5 minutes

                    target_id = remaining_targets[i]
                    futures[
                        executor.submit(process_target_with_memory_check, target_id)
                    ] = target_id

                # Process remaining targets as tasks complete
                next_target_idx = batch_size

                for future in as_completed(futures):
                    target_id = futures[future]
                    try:
                        result = future.result()
                        print(result)
                    except Exception as exc:
                        print(f"{target_id} generated an exception: {exc}")

                    # Submit next task if available
                    if next_target_idx < len(remaining_targets):
                        # Check memory before submitting next task
                        while check_memory_usage(threshold_percent=85):
                            print(
                                "Memory usage too high, waiting 5 minutes before submitting more tasks..."
                            )
                            time.sleep(300)  # Sleep for 5 minutes

                        next_target = remaining_targets[next_target_idx]
                        futures[
                            executor.submit(
                                process_target_with_memory_check, next_target
                            )
                        ] = next_target
                        next_target_idx += 1

# Formatting Evaluation Metrics


In [6]:
def segregate_identifier(
    index,
    column_return_type,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    ind_target_grp,
    num_pop_folds,
    num_ind_folds,
):
    index = str(index).upper()
    if column_return_type == "Pop_Epoch_Size":
        for i in POPULATION_EPOCH_SIZE_LIST:
            if str(i) + "_POP" in index:
                return i
    if column_return_type == "Ind_Epoch_Size":
        for j in PATIENT_EPOCH_SIZE_LIST:
            if str(j) + "_IND" in index:
                return j
    elif column_return_type == "Patient_ID":
        for i in ind_target_grp:
            if str(i) in index:
                return i
    elif column_return_type == "Pop_Fold":
        for i in [*range(1, num_pop_folds + 1)]:
            if "POP_FOLD_" + str(i) in index:
                return i
    elif column_return_type == "Ind_Fold":
        for i in [*range(1, num_ind_folds + 1)]:
            if "IND_FOLD_" + str(i) in index:
                return i


def load_csv_to_df(
    file_path,
    data_processing_columns,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    ind_target_grp,
    num_pop_folds,
    num_ind_folds,
):
    df = pd.read_csv(file_path)
    index_col_name = df.columns[0]
    df.drop_duplicates(subset=[index_col_name], keep="last", inplace=True)
    df.set_index(index_col_name, inplace=True)
    for i in data_processing_columns:
        df[i] = df.index.map(
            lambda x: segregate_identifier(
                x,
                i,
                POPULATION_EPOCH_SIZE_LIST,
                PATIENT_EPOCH_SIZE_LIST,
                ind_target_grp,
                num_pop_folds,
                num_ind_folds,
            )
        )

    return df


def resave_eval_df(
    eval_data_type,
    RESULT_DIR,
    Architecture_GROUP,
    bool_remove_target,
    data_processing_columns,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    ind_target_grp,
    num_pop_folds,
    num_ind_folds,
):
    if bool_remove_target == True:
        suffix = "_evaluation_metrics_TargetRemoved.csv"
    else:
        suffix = "_evaluation_metrics.csv"

    if eval_data_type == "individual":
        eval_data_path = "/diagrams_n_individual_models/Ind_"
    elif eval_data_type == "population":
        eval_data_path = "/diagrams_n_population_models/Pop_"

    file_path = (
        str(RESULT_DIR)
        + "/"
        + Architecture_GROUP
        + eval_data_path
        + Architecture_GROUP
        + suffix
    )

    if os.path.isfile(file_path):
        temp_unsorted_eval_data_df = load_csv_to_df(
            file_path,
            data_processing_columns,
            POPULATION_EPOCH_SIZE_LIST,
            PATIENT_EPOCH_SIZE_LIST,
            ind_target_grp,
            num_pop_folds,
            num_ind_folds,
        )

        temp_sorted_eval_data_df = temp_unsorted_eval_data_df.sort_values(
            by=data_processing_columns, ascending=True
        ).drop(columns=data_processing_columns, errors="ignore")

        output_path = os.path.join(file_path)
        temp_sorted_eval_data_df.to_csv(
            output_path, mode="w", index=True, index_label=""
        )


data_processing_columns = [
    "Patient_ID",
    "Pop_Epoch_Size",
    "Pop_Fold",
    "Ind_Epoch_Size",
    "Ind_Fold",
]

bool_remove_target_list = [True, False]
eval_data_type_list = ["individual", "population"]
for i in eval_data_type_list:
    for j in bool_remove_target_list:
        resave_eval_df(
            i,
            RESULT_DIR,
            Architecture_GROUP,
            j,
            data_processing_columns,
            POPULATION_EPOCH_SIZE_LIST,
            PATIENT_EPOCH_SIZE_LIST,
            ind_target_grp,
            num_pop_folds,
            num_ind_folds,
        )

del data_processing_columns
del bool_remove_target_list
del eval_data_type_list

# End
